# 3.1  Basic Altair scatter plot
The minimal Altair chart is three lines: data, mark, and encoding. Everything else is optional.

In [1]:
import altair as alt
from vega_datasets import data as vdata

# Load the Gapminder dataset via vega_datasets
source = vdata.gapminder_health_income()

# alt.Chart(data)            -> binds the dataset to this chart
# .mark_point(filled=True)   -> draw a filled circle for each row
# .encode(...)               -> map columns to visual channels

chart = (
    alt.Chart(source)
    .mark_point(filled=True, size=80)
    .encode(
        x=alt.X(
            'income:Q',                  # :Q = quantitative (continuous number)
            scale=alt.Scale(type='log'),  # log scale on the x axis
            title='Income per person (log scale)'
        ),
        y=alt.Y(
            'health:Q',
            scale=alt.Scale(zero=False),  # don't force y axis to start at 0
            title='Life expectancy (years)'
        ),
        color=alt.Color(
            'region:N',                   # :N = nominal (unordered category)
            title='Region'
        ),
        size=alt.Size(
            'population:Q',
            scale=alt.Scale(range=[50, 1000]),  # min and max pixel area
            title='Population'
        ),
        tooltip=[                         # columns shown on hover
            alt.Tooltip('country:N', title='Country'),
            alt.Tooltip('income:Q', title='Income', format=',.0f'),
            alt.Tooltip('health:Q', title='Life exp', format='.1f'),
        ]
    )
    .properties(
        width=580,
        height=380,
        title='Income vs health by country'
    )
    .interactive()  # enables pan and zoom with scroll/drag
)

chart

alt.Chart(...)

In [2]:
import pandas as pd
from vega_datasets import data as vdata

source = vdata.gapminder_health_income()
print(source.columns)

Index(['country', 'income', 'health', 'population'], dtype='object')


In [3]:
import altair as alt
from vega_datasets import data as vdata

source = vdata.gapminder_health_income()

chart = (
    alt.Chart(source)
    .mark_point(filled=True, size=80)
    .encode(
        x=alt.X('income:Q', scale=alt.Scale(type='log')),
        y='health:Q',

        # create meaningful grouping instead of country
        color=alt.Color(
            'income:Q',
            bin=alt.Bin(maxbins=5),
            title='Income Group'
        ),

        size='population:Q',
        tooltip=['country:N', 'income:Q', 'health:Q', 'population:Q']
    )
    .properties(width=580, height=380, title='Income vs Health')
    .interactive()
)

chart

alt.Chart(...)

In [4]:
import altair as alt
from vega_datasets import data as vdata

# Load dataset
source = vdata.gapminder_health_income()

chart = (
    alt.Chart(source)
    .mark_point(filled=True, size=80)
    .encode(
        x=alt.X(
            'income:Q',
            scale=alt.Scale(type='log'),
            title='Income per person (log scale)'
        ),
        y=alt.Y(
            'health:Q',
            scale=alt.Scale(zero=False),
            title='Life expectancy (years)'
        ),

        # FIX: use country instead of region
        color=alt.Color('country:N', title='Country'),

        size=alt.Size(
            'population:Q',
            scale=alt.Scale(range=[50, 1000]),
            title='Population'
        ),

        tooltip=[
            alt.Tooltip('country:N', title='Country'),
            alt.Tooltip('income:Q', title='Income', format=',.0f'),
            alt.Tooltip('health:Q', title='Life exp', format='.1f'),
            alt.Tooltip('population:Q', title='Population', format=',.0f')
        ]
    )
    .properties(
        width=580,
        height=380,
        title='Income vs Health by Country'
    )
    .interactive()
)

chart

alt.Chart(...)

# 3.2  Bar chart with inline aggregation
Altair can aggregate data inside the encoding specification, so you do not need to pre
aggregate in pandas.

In [7]:
import altair as alt
from vega_datasets import data as vdata

source = vdata.gapminder_health_income()

# Create income groups (acts like region replacement)
source['income_group'] = pd.qcut(source['income'], q=4, labels=[
    'Low Income', 'Lower-Mid', 'Upper-Mid', 'High Income'
])

chart = (
    alt.Chart(source)
    .mark_bar()
    .encode(
        x=alt.X('mean(health):Q', title='Average life expectancy'),

        # FIX: use derived grouping instead of region
        y=alt.Y(
            'income_group:N',
            sort='-x',
            title=''
        ),

        color=alt.Color('income_group:N', legend=None),

        tooltip=[
            alt.Tooltip('income_group:N', title='Income Group'),
            alt.Tooltip('mean(health):Q', title='Avg life exp', format='.1f'),
        ]
    )
    .properties(
        width=520,
        height=300,
        title='Average life expectancy by income group'
    )
)

chart

alt.Chart(...)

# 3.3  Linked brush selection
This is Altair's flagship feature. A selection defined on one chart automatically filters another 
chart in the same specification. No callbacks or JavaScript required.

In [9]:
import altair as alt
from vega_datasets import data as vdata

source = vdata.gapminder_health_income()

# 1. Brush selection (drag rectangle)
brush = alt.selection_interval(encodings=['x', 'y'])

# 2. Scatter plot
scatter = (
    alt.Chart(source)
    .mark_point(filled=True)
    .encode(
        x=alt.X(
            'income:Q',
            scale=alt.Scale(type='log'),
            title='Income (log)'
        ),
        y=alt.Y(
            'health:Q',
            scale=alt.Scale(zero=False),
            title='Life expectancy'
        ),

        # FIX: replace region with country (or remove color grouping)
        color=alt.condition(
            brush,
            alt.value('steelblue'),   # selected points
            alt.value('lightgrey')    # unselected points
        ),

        size=alt.condition(
            brush,
            alt.value(80),
            alt.value(30)
        ),

        tooltip=[
            alt.Tooltip('country:N'),
            alt.Tooltip('income:Q'),
            alt.Tooltip('health:Q')
        ]
    )
    .add_params(brush)
    .properties(
        width=380,
        height=300,
        title='Click and drag to select'
    )
)

# 3. Histogram of selected points
histogram = (
    alt.Chart(source)
    .mark_bar()
    .encode(
        x=alt.X('income:Q', bin=alt.Bin(maxbins=20), title='Income'),
        y=alt.Y('count()', title='Countries')
    )
    .transform_filter(brush)
    .properties(
        width=380,
        height=300,
        title='Distribution of selected countries'
    )
)

# 4. Combine charts
scatter | histogram

alt.HConcatChart(...)

# 3.4  Faceted chart (small multiples)
Faceting replicates the same chart for each category, making it easy to compare distributions 
without overplotting.

In [10]:
import altair as alt
from vega_datasets import data as vdata
import pandas as pd

source = vdata.gapminder_health_income()

# Create categorical income groups for faceting
source['income_group'] = pd.qcut(
    source['income'],
    q=3,
    labels=['Low Income', 'Mid Income', 'High Income']
)

chart = (
    alt.Chart(source)
    .mark_point(filled=True, opacity=0.7)
    .encode(
        x=alt.X(
            'income:Q',
            scale=alt.Scale(type='log'),
            title='Income (log)'
        ),
        y=alt.Y(
            'health:Q',
            scale=alt.Scale(zero=False),
            title='Life expectancy'
        ),
        size=alt.Size('population:Q', legend=None),
        tooltip=[
            alt.Tooltip('country:N'),
            alt.Tooltip('income:Q'),
            alt.Tooltip('health:Q')
        ]
    )
    .properties(width=220, height=180)
    .facet(
        facet=alt.Facet('income_group:N', title='Income Group'),
        columns=3
    )
    .resolve_scale(y='independent')
)

chart

alt.FacetChart(...)